In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pickle
import os

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
print("✅ Library siap")

In [ ]:
train_df = pd.read_csv('../data/aug_train.csv')
test_df  = pd.read_csv('../data/aug_test.csv')

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

In [ ]:
def preprocess_v3_fe(df, is_train=True):
    df = df.copy()
    df.drop(columns=['enrollee_id', 'city'], inplace=True)

    if is_train:
        y = df.pop('target').astype(int)

    # 1. Missing Indicator Flags
    flag_cols = ['gender', 'company_size', 'company_type',
                 'major_discipline', 'last_new_job', 'enrolled_university']
    for col in flag_cols:
        df[f'{col}_missing'] = df[col].isnull().astype(int)

    # 2. Imputasi
    mode_cols = ['gender', 'enrolled_university', 'education_level',
                 'major_discipline', 'experience', 'last_new_job']
    for col in mode_cols:
        df[col] = df[col].fillna(df[col].mode()[0])
    df['company_size'] = df['company_size'].fillna('Unknown')
    df['company_type'] = df['company_type'].fillna('Unknown')

    # 3. Feature Engineering
    df['job_hopper'] = (
        (df['last_new_job'].isin(['1', 'never'])) &
        (df['experience'].isin(['<1','1','2','3','4','5']))
    ).astype(int)

    df['high_value_candidate'] = (
        (df['experience'].isin(['10','11','12','13','14','15','>20'])) &
        (df['education_level'].isin(['Masters', 'Phd']))
    ).astype(int)

    df['is_fresher'] = (
        (df['experience'].isin(['<1', '1', '2'])) &
        (df['enrolled_university'] != 'no_enrollment')
    ).astype(int)

    df['mismatch_candidate'] = (
        (df['relevent_experience'] == 'No relevent experience') &
        (df['company_size'].isin(['<10', '10/49', '50-99', 'Unknown']))
    ).astype(int)

    df['ambitious'] = (
        (df['city_development_index'] < 0.75) &
        (df['training_hours'] > 50)
    ).astype(int)

    df['career_stagnant'] = (
        (df['last_new_job'].isin(['>4', '4'])) &
        (df['experience'].isin(['10','11','12','13','14','15','>20']))
    ).astype(int)

    exp_map = {'<1': 0.5, '>20': 21}
    exp_numeric = df['experience'].replace(exp_map)
    exp_numeric = pd.to_numeric(exp_numeric, errors='coerce').fillna(1)
    df['training_per_exp'] = df['training_hours'] / (exp_numeric + 1)

    df['city_tier'] = pd.cut(
        df['city_development_index'],
        bins=[0, 0.62, 0.78, 0.90, 1.0],
        labels=[0, 1, 2, 3]
    ).astype(int)

    # 4. Ordinal Encoding
    ordinal_config = {
        'experience'     : ['<1','1','2','3','4','5','6','7','8','9','10',
                            '11','12','13','14','15','16','17','18','19','20','>20'],
        'last_new_job'   : ['never','1','2','3','4','>4'],
        'company_size'   : ['<10','10/49','50-99','100-500','500-999',
                            '1000-4999','5000-9999','10000+','Unknown'],
        'education_level': ['Primary School','High School','Graduate','Masters','Phd'],
    }
    for col, order in ordinal_config.items():
        enc = OrdinalEncoder(
            categories=[order],
            handle_unknown='use_encoded_value',
            unknown_value=-1
        )
        df[col] = enc.fit_transform(df[[col]])

    # 5. Nominal Encoding
    nominal_cols = ['gender', 'relevent_experience', 'enrolled_university',
                    'major_discipline', 'company_type']
    for col in nominal_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

    if is_train:
        return df, y
    return df

X, y = preprocess_v3_fe(train_df, is_train=True)
print(f"✅ Total fitur: {X.shape[1]} kolom")

In [ ]:
# Split Data
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# SMOTE untuk imbalanced data
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

# STANDARD SCALER (Tahapan Paling Krusial untuk SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_val_scaled = scaler.transform(X_val)

print("✅ Data selesai discale dan di-SMOTE")

In [ ]:
param_grid = {
    'C': [0.1, 1.0],
    'kernel': ['rbf'] 
}

# probability=True penting agar kita bisa pakai predict_proba() di Web Streamlit nanti
base_svm = SVC(probability=True, random_state=42)

grid_search = GridSearchCV(base_svm, param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=2)
grid_search.fit(X_train_scaled, y_train_res)

best_svm = grid_search.best_estimator_

print(f"Parameter Terbaik: {grid_search.best_params_}")
print("✅ Model SVM selesai ditraining")

In [ ]:
y_pred_svm  = best_svm.predict(X_val_scaled)
y_proba_svm = best_svm.predict_proba(X_val_scaled)[:, 1]

print("=== EVALUASI SUPPORT VECTOR MACHINE ===")
print(classification_report(y_val, y_pred_svm))
print(f"ROC-AUC : {roc_auc_score(y_val, y_proba_svm):.4f}")

In [ ]:
# Pastikan folder models/ sudah ada
os.makedirs('../models', exist_ok=True)

# Simpan Model
with open('../models/svm_model.pkl', 'wb') as f:
    pickle.dump(best_svm, f)

# Simpan Scaler
with open('../models/svm_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("✅ Model dan Scaler berhasil diekspor ke folder models/")